In [1]:
# ProcessMine Multi-Model Comparative Analysis
# Comprehensive evaluation of available models

import os
import logging
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Import ProcessMine modules
from processmine.data.loader import load_and_preprocess_data
from processmine.models.factory import create_model, get_model_config
from processmine.core.runner import run_training



2025-03-19 01:22:01,341 - INFO: PM4Py not available. Using simplified conformance checking.


In [2]:
def safe_model_creation(model_type, **kwargs):
    """
    Safely create models with robust error handling and configuration

    Args:
        model_type: Type of model to create
        **kwargs: Configuration parameters

    Returns:
        Created model instance
    """
    try:
        # Ensure required parameters are present
        model_config = get_model_config(model_type).copy()
        model_config.update(kwargs)

        # Special handling for different model types
        if model_type in ['gnn', 'enhanced_gnn', 'positional_gnn', 'diverse_gnn']:
            # Ensure consistent GNN model creation
            required_keys = ['input_dim', 'output_dim']
            for key in required_keys:
                if key not in model_config:
                    raise ValueError(f"{key} is required for {model_type} model")

            # Adjust attention type based on model type
            if model_type == 'enhanced_gnn':
                model_config['attention_type'] = 'combined'
            elif model_type == 'positional_gnn':
                model_config['attention_type'] = 'positional'
            elif model_type == 'diverse_gnn':
                model_config['attention_type'] = 'diverse'

        elif model_type in ['lstm', 'enhanced_lstm']:
            # LSTM models don't use input_dim parameter
            if 'input_dim' in model_config:
                model_config.pop('input_dim')
                
            # Ensure output_dim is converted to num_cls
            if 'output_dim' in model_config and 'num_cls' not in model_config:
                model_config['num_cls'] = model_config.pop('output_dim')

        elif model_type in ['decision_tree', 'random_forest', 'xgboost']:
            # For tree-based models, ensure class weight handling
            model_config['class_weight'] = model_config.get('class_weight', 'balanced')

        # Create and return model
        return create_model(model_type, **model_config)

    except Exception as e:
        logger.error(f"Failed to create {model_type} model: {e}")
        import traceback
        traceback.print_exc()
        return None


In [3]:
def run_model_comparative_analysis(data_path):
    """
    Comprehensive model comparison across different model types.

    Args:
        data_path: Path to input CSV file.

    Returns:
        Dictionary of model results.
    """
    # 1. Data Loading and Preprocessing
    logger.info("Starting data loading and preprocessing...")
    df, task_encoder, resource_encoder = load_and_preprocess_data(
        data_path,
        norm_method='l2',
        use_dtypes=True,
        memory_limit_gb=4.0,
        cache_dir="results/cache"
    )

    # Prepare feature columns
    feature_cols = [col for col in df.columns if col.startswith("feat_")]

    # Prepare features and targets
    X = df[feature_cols].values
    y = df['next_task'].values

    # CRITICAL FIX FOR XGBOOST: Ensure class labels are contiguous integers
    # XGBoost requires classes 0,1,2,3... with no gaps
    unique_classes = np.unique(y)
    class_map = {old_cls: new_cls for new_cls, old_cls in enumerate(unique_classes)}
    y_mapped = np.array([class_map[cls] for cls in y])
    
    # Store reverse mapping for interpretation
    reverse_map = {new_cls: old_cls for old_cls, new_cls in class_map.items()}

    # Split data AFTER applying the encoding
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_mapped, test_size=0.2, random_state=42
    )

    # Prepare input and output dimensions
    input_dim = X.shape[1]
    output_dim = len(unique_classes)

    # Models to test
    MODELS_TO_TEST = [
        ('gnn', {
            'input_dim': input_dim,
            'output_dim': output_dim,
            'hidden_dim': 64,
            'num_layers': 2,
            'dropout': 0.3,
            'attention_type': 'basic'
        }),
        ('enhanced_gnn', {
            'input_dim': input_dim,
            'output_dim': output_dim,
            'hidden_dim': 64,
            'num_layers': 2,
            'dropout': 0.3,
            'attention_type': 'combined'
        }),
        ('lstm', {
            # Don't include input_dim for LSTM models
            'output_dim': output_dim,
            'emb_dim': 64,
            'hidden_dim': 64,
            'num_layers': 2,
            'dropout': 0.3
        }),
        ('decision_tree', {
            'max_depth': 10,
            'min_samples_split': 5,
            'class_weight': 'balanced'
        }),
        ('random_forest', {
            'n_estimators': 100,
            'max_depth': 10,
            'class_weight': 'balanced'
        }),
        ('xgboost', {
            'n_estimators': 100,
            'max_depth': 10,
            'learning_rate': 0.1,
            'objective': 'multi:softmax',
            'num_class': output_dim
        })
    ]

    # Results storage
    model_results = {}

    # Evaluation function
    def evaluate_model(model, X_test, y_test):
        """Evaluate model and return metrics"""
        from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

        # Predict
        y_pred = model.predict(X_test)

        # Calculate metrics
        return {
            'accuracy': accuracy_score(y_test, y_pred),
            'f1_weighted': f1_score(y_test, y_pred, average='weighted'),
            'precision': precision_score(y_test, y_pred, average='weighted'),
            'recall': recall_score(y_test, y_pred, average='weighted')
        }

    # 2. Model Training and Evaluation
    for model_name, model_config in MODELS_TO_TEST:
        try:
            logger.info(f"Training {model_name} model...")

            # Create model
            model = safe_model_creation(model_name, **model_config)

            if model is None:
                logger.error(f"Failed to create {model_name} model")
                continue

            # Train model (with special handling for different types)
            if model_name in ['gnn', 'enhanced_gnn', 'lstm']:
                # For neural models that use run_training
                training_configs = model_config.copy()
                
                # For GNN models, fix input_dim passing
                if model_name in ['gnn', 'enhanced_gnn']:
                    # Make sure run_training gets input_dim
                    training_configs = {**training_configs, 'input_dim': input_dim}
                
                # For LSTM models, remove input_dim if present
                if model_name == 'lstm' and 'input_dim' in training_configs:
                    training_configs.pop('input_dim')
                
                # Use run_training for neural models
                training_results = run_training(
                    data_path=data_path,
                    model=model_name,
                    **training_configs
                )

                # Extract model from results if possible
                if isinstance(training_results, dict) and 'model' in training_results:
                    model = training_results['model']
                
                # For neural models, use placeholder metrics for now
                metrics = {'accuracy': 0.0, 'f1_weighted': 0.0, 'precision': 0.0, 'recall': 0.0}

            elif model_name in ['decision_tree', 'random_forest']:
                # Directly fit for tree-based models
                model.fit(X_train, y_train)
                
                # Evaluate model
                metrics = evaluate_model(model, X_test, y_test)
                
            elif model_name == 'xgboost':
                # Double-check that training labels are contiguous
                train_classes = np.unique(y_train)
                expected_classes = np.arange(len(train_classes))
                
                if not np.array_equal(train_classes, expected_classes):
                    logger.warning(f"XGBoost train classes not contiguous: {train_classes}")
                    # Remap train and test data to ensure contiguity
                    train_map = {old: new for new, old in enumerate(sorted(np.unique(y_train)))}
                    y_train = np.array([train_map[y] for y in y_train])
                    y_test = np.array([train_map.get(y, 0) for y in y_test])  # Default to 0 for unseen classes
                
                # Update num_class parameter for XGBoost
                if hasattr(model, 'get_params') and 'num_class' in model.get_params():
                    model.set_params(num_class=len(np.unique(y_train)))
                
                # Now fit the model
                model.fit(X_train, y_train)
                
                # Evaluate model
                metrics = evaluate_model(model, X_test, y_test)

            # Store results
            model_results[model_name] = {
                'model': model,
                'metrics': metrics
            }

            logger.info(f"{model_name} training completed. Performance: {metrics}")

        except Exception as e:
            logger.error(f"Error training {model_name}: {e}")
            import traceback
            traceback.print_exc()

    # 3. Visualization of Results
    if model_results:
        # Prepare comparison data
        comparison_data = []
        for model_name, result in model_results.items():
            metrics = result['metrics']
            comparison_data.append({
                'model': model_name,
                'accuracy': metrics['accuracy'],
                'f1_weighted': metrics['f1_weighted']
            })

        # Create DataFrame
        comparison_df = pd.DataFrame(comparison_data)

        # Plot results
        plt.figure(figsize=(10, 6))
        comparison_df.plot(x='model', y=['accuracy', 'f1_weighted'], kind='bar')
        plt.title('Model Performance Comparison')
        plt.xlabel('Model')
        plt.ylabel('Score')
        plt.tight_layout()
        plt.savefig('results/model_performance_comparison.png')
        plt.close()

    # 4. Generate Summary Report
    os.makedirs('results', exist_ok=True)
    with open('results/model_comparison_summary.txt', 'w') as f:
        f.write("ProcessMine Multi-Model Comparative Analysis\n")
        f.write("=" * 50 + "\n\n")

        for model_name, result in model_results.items():
            f.write(f"Model: {model_name}\n")
            metrics = result['metrics']
            for metric, value in metrics.items():
                f.write(f"  {metric.capitalize()}: {value:.4f}\n")
            f.write("\n")

    return model_results

In [4]:
# Main Execution
if __name__ == "__main__":
    data_path = "../input/BPI2020_DomesticDeclarations.csv"
    
    try:
        # Run comprehensive model analysis
        results = run_model_comparative_analysis(data_path)
        
        print("Model comparison completed successfully!")
        print(f"Results available for {len(results)} models.")
    except Exception as e:
        logger.error(f"Comprehensive model analysis failed: {e}")
        import traceback
        traceback.print_exc()

2025-03-19 01:22:01,423 - INFO: Starting data loading and preprocessing...
2025-03-19 01:22:01,425 - INFO: Loading data from ../input/BPI2020_DomesticDeclarations.csv
2025-03-19 01:22:01,426 - WARNING: Failed to load cached data: [Errno 2] No such file or directory: 'results/cache/processed_-6356520965788638562.pkl'
2025-03-19 01:22:01,427 - INFO: File size: 0.01 GB, loading in single pass
2025-03-19 01:22:01,881 - INFO: Loaded 56,437 rows in single pass
2025-03-19 01:22:02,004 - INFO: Removed 10,500 end events (no next task)
2025-03-19 01:22:02,396 - INFO: Cached preprocessed data to results/cache/processed_-6356520965788638562.pkl
2025-03-19 01:22:02,397 - INFO: Preprocessing completed in 0.97s
2025-03-19 01:22:02,397 - INFO: Data statistics:
2025-03-19 01:22:02,403 - INFO:   Cases: 10,366
2025-03-19 01:22:02,404 - INFO:   Activities: 17
2025-03-19 01:22:02,405 - INFO:   Resources: 2
2025-03-19 01:22:02,405 - INFO:   Events: 45,937
2025-03-19 01:22:02,406 - INFO:   Time range: 2017-0

Model comparison completed successfully!
Results available for 5 models.


<Figure size 1000x600 with 0 Axes>